# Marketing Performance and Measurement Analysis
### Synthetic campaign data | June–August 2026

## Business objective

Evaluate campaign performance across Google Ads, Meta Ads, TikTok Ads and DV360, and investigate whether measurement-source differences affect performance assessment.

The analysis addresses three questions:

1. How does platform performance change when campaigns are compared within the same funnel stage?
2. Which metrics help explain differences in conversion campaign CPA and ROAS?
3. When do platform and analytics-source results diverge, and how does this affect reported ROAS?

## Dataset

The dataset contains 6,624 records covering 24 campaigns from 1 June to 31 August 2026. Each row represents one date, campaign and device combination.

All data is synthetic and created for portfolio purposes. Results should not be interpreted as real advertiser results or industry benchmarks.

## Tools

Python and pandas are used for data preparation and validation. SQL with DuckDB is used for performance analysis and measurement reconciliation. Looker Studio presents the results through three interactive dashboard pages: Performance Overview, Conversion Diagnostics and Measurement Reconciliation.

## 1. Environment setup and data loading

This notebook runs in Google Colab and loads the synthetic CSV
directly from the public GitHub repository.

Run all cells from top to bottom. No Google Drive setup or
personal file-path changes are required.

In [17]:

%pip install -q duckdb

In [18]:
from pathlib import Path
import pandas as pd

# Load the CSV from the public GitHub repository
data_url = (
    "https://raw.githubusercontent.com/"
    "Marcel01205/marketing-performance-analytics/main/"
    "data/marketing_performance_raw_jun_aug_2026.csv"
)

df = pd.read_csv(data_url)
df["Date"] = pd.to_datetime(df["Date"])

print("Rows loaded:", len(df))
print("Columns:", df.shape[1])
print("Campaigns:", df["Campaign_ID"].nunique())
print(
    "Date range:",
    df["Date"].min().date(),
    "to",
    df["Date"].max().date()
)

Rows loaded: 6624
Columns: 23
Campaigns: 24
Date range: 2026-06-01 to 2026-08-31


## 2. Data preparation and validation

Before calculating performance metrics, I inspect missing values, duplicate keys, negative values and campaign-day consistency.

`Daily_Budget_EUR` is a campaign-day value repeated across device rows and must be counted once per campaign-day. `Spend_EUR` is device-level spend and can be summed across devices.

During preparation, repeated spend values were identified across devices. The synthetic dataset was revised to allocate campaign-day spend across devices while respecting the requested daily budget cap. This dataset-generation correction also changed cost-based metrics and ROAS.

Budget comparisons use integer cents to avoid floating-point precision artifacts.

In [19]:
print("Data rows:", len(df))
print("Columns:", df.shape[1])
print(
    "Date range:",
    df["Date"].min().date(),
    "to",
    df["Date"].max().date()
)

print(
    "Duplicate date-campaign-device rows:",
    df.duplicated(["Date", "Campaign_ID", "Device"]).sum()
)

print("\nMissing values by column:")
display(df.isna().sum().sort_values(ascending=False))

print("\nColumn types:")
display(df.dtypes)

print("\nSample records:")
display(
    df[
        [
            "Date",
            "Platform",
            "Campaign_ID",
            "Device",
            "Daily_Budget_EUR",
            "Spend_EUR"
        ]
    ].head(6).style.format({
        "Date": "{:%Y-%m-%d}",
        "Daily_Budget_EUR": "€{:,.2f}",
        "Spend_EUR": "€{:,.2f}"
    })
)

Data rows: 6624
Columns: 23
Date range: 2026-06-01 to 2026-08-31
Duplicate date-campaign-device rows: 0

Missing values by column:


,0
Date,0
Platform,0
Campaign_ID,0
Campaign_Name,0
Campaign_Type,0
Objective,0
Audience,0
Creative,0
Buying_Strategy,0
Device,0



Column types:


,0
Date,datetime64[ns]
Platform,object
Campaign_ID,object
Campaign_Name,object
Campaign_Type,object
Objective,object
Audience,object
Creative,object
Buying_Strategy,object
Device,object



Sample records:


,Date,Platform,Campaign_ID,Device,Daily_Budget_EUR,Spend_EUR
0,2026-06-01,Google Ads,GGL_AW01,Mobile,"€1,400.00",€474.67
1,2026-06-01,Google Ads,GGL_AW01,Desktop,"€1,400.00",€408.83
2,2026-06-01,Google Ads,GGL_AW01,Tablet,"€1,400.00",€516.50
3,2026-06-02,Google Ads,GGL_AW01,Mobile,"€1,400.00",€479.27
4,2026-06-02,Google Ads,GGL_AW01,Desktop,"€1,400.00",€399.32
5,2026-06-02,Google Ads,GGL_AW01,Tablet,"€1,400.00",€521.41


In [20]:
# Aggregate device-level spend to campaign-day totals and count device rows
campaign_daily = df.groupby(["Date", "Campaign_ID"]).agg(
    Total_Spend_EUR=("Spend_EUR", "sum"),
    Daily_Budget_EUR=("Daily_Budget_EUR", "first"),
    Device_Row_Count=("Device", "count")
).reset_index()

print("Campaign-days:", len(campaign_daily))
print(
    "Campaign-days without exactly three device rows:",
    (campaign_daily["Device_Row_Count"] != 3).sum()
)

Campaign-days: 2208
Campaign-days without exactly three device rows: 0


In [21]:
# Compare money as integer cents to avoid floating-point artifacts
campaign_daily["Spend_Cents"] = (
    campaign_daily["Total_Spend_EUR"] * 100
).round().astype("int64")

campaign_daily["Budget_Cents"] = (
    campaign_daily["Daily_Budget_EUR"] * 100
).round().astype("int64")

budget_exceeded = campaign_daily[
    campaign_daily["Spend_Cents"] > campaign_daily["Budget_Cents"]
]

print("Campaign-days exceeding budget:", len(budget_exceeded))

metrics = [
    "Spend_EUR", "Impressions", "Clicks",
    "Platform_Conversions", "Analytics_Conversions",
    "Platform_Revenue_EUR", "Analytics_Revenue_EUR"
]

for metric in metrics:
    print(f"Negative values in {metric}: {(df[metric] < 0).sum()}")

Campaign-days exceeding budget: 0
Negative values in Spend_EUR: 0
Negative values in Impressions: 0
Negative values in Clicks: 0
Negative values in Platform_Conversions: 0
Negative values in Analytics_Conversions: 0
Negative values in Platform_Revenue_EUR: 0
Negative values in Analytics_Revenue_EUR: 0


In [22]:
# 1. Verify if the 3 rows per day/campaign represent 3 distinct devices
device_counts = df.groupby(['Date', 'Campaign_ID'])['Device'].nunique()
all_distinct_devices = (device_counts == 3).all()
print(f"Does every combination of day and campaign have exactly 3 distinct devices? {all_distinct_devices}")
if not all_distinct_devices:
    print(f"Number of combinations that do not have 3 distinct devices: {(device_counts != 3).sum()}")

# 2. Verify if Daily_Budget_EUR has the exact same value across all rows of each daily campaign
budget_diversity = df.groupby(['Date', 'Campaign_ID'])['Daily_Budget_EUR'].nunique()
same_budget_everywhere = (budget_diversity == 1).all()
print(f"Does 'Daily_Budget_EUR' have the exact same value across all rows for each day/campaign? {same_budget_everywhere}")
if not same_budget_everywhere:
    print(f"Number of combinations with inconsistent budgets: {(budget_diversity > 1).sum()}")

Does every combination of day and campaign have exactly 3 distinct devices? True
Does 'Daily_Budget_EUR' have the exact same value across all rows for each day/campaign? True


### Validation outcome

The corrected dataset contains 6,624 records and 2,208 campaign-day combinations.

The checks found no missing values, duplicate date-campaign-device keys or negative values in the tested performance metrics. Every campaign-day contains three distinct devices and one consistent daily budget. No campaign-day exceeds the budget cap when monetary values are compared in cents.

These checks establish structural consistency. They do not validate attribution accuracy or the commercial realism of the synthetic results.

## 3. SQL analysis setup

DuckDB queries the validated pandas DataFrame directly. The first query confirms the record and campaign counts.

Performance ratios are calculated from aggregated totals, rather than averaging row-level ratios. `NULLIF` prevents division by zero.

Platform and analytics-source conversions and revenue are kept separate. CVR is defined here as reported conversions divided by clicks.

This CVR is a reported-conversions-to-clicks ratio. Without attribution-level records, it should not be interpreted as a verified post-click conversion rate.

In [23]:
import duckdb

result = duckdb.sql("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT Campaign_ID) AS campaign_count
    FROM df
""").df()

display(result)

,row_count,campaign_count
0,6624,24


## 4. Platform performance overview

**Question:** How are spend, reported conversions and attributed revenue distributed across platforms?

This comparison includes all funnel stages. It provides an overview, but campaign objectives and attribution differences must be considered when interpreting the ranking.

In [24]:
# SQL query to analyze data by platform

platform_analysis = duckdb.sql("""
    SELECT
        Platform,
        SUM(Spend_EUR) AS Total_Spend_EUR,
        SUM(Platform_Conversions) AS Total_Platform_Conversions,
        SUM(Platform_Revenue_EUR) AS Total_Platform_Revenue_EUR,
        -- CPA = total spend ÷ total conversions
        SUM(Spend_EUR) / NULLIF(SUM(Platform_Conversions), 0) AS CPA,
        -- ROAS = total revenue ÷ total spend
        SUM(Platform_Revenue_EUR) / NULLIF(SUM(Spend_EUR), 0) AS ROAS
    FROM df
    GROUP BY Platform
    ORDER BY Platform
""").df()

display(
    platform_analysis.style.format({
        "Total_Spend_EUR": "€{:,.2f}",
        "Total_Platform_Conversions": "{:,.0f}",
        "Total_Platform_Revenue_EUR": "€{:,.2f}",
        "CPA": "€{:.2f}",
        "ROAS": "{:.2f}x"
    })
)

,Platform,Total_Spend_EUR,Total_Platform_Conversions,Total_Platform_Revenue_EUR,CPA,ROAS
0,DV360,"€721,004.05","45,795","€6,791,480.74",€15.74,9.42x
1,Google Ads,"€614,261.95","97,557","€13,312,296.02",€6.30,21.67x
2,Meta Ads,"€621,213.03","108,377","€15,511,163.75",€5.73,24.97x
3,TikTok Ads,"€535,421.88","53,766","€6,006,914.96",€9.96,11.22x


### Interpretation

Meta Ads has the lowest aggregate CPA (€5.73) and the highest platform-reported ROAS (24.97). DV360 has the highest aggregate CPA (€15.74) and the lowest ROAS (9.42).

The next comparison separates funnel stages to assess how campaign composition affects these results.

## 5. Performance by funnel stage

**Question:** Does the platform ranking change when campaigns are compared within the same funnel stage?

I group results by platform and campaign type and count distinct campaigns. CPA and ROAS alone are insufficient to assess Awareness performance.

In [25]:
# Analysis of data by platform and campaign type (funnel stage)
funnel_analysis = duckdb.sql("""
    SELECT
        Platform,
        Campaign_Type,
        COUNT(DISTINCT Campaign_ID) AS Campaign_Count,
        SUM(Spend_EUR) AS Total_Spend_EUR,
        SUM(Platform_Conversions) AS Total_Platform_Conversions,
        SUM(Platform_Revenue_EUR) AS Total_Platform_Revenue_EUR,
        -- CPA = total spend ÷ total conversions
        SUM(Spend_EUR) / NULLIF(SUM(Platform_Conversions), 0) AS CPA,
        -- ROAS = total revenue ÷ total spend
        SUM(Platform_Revenue_EUR) / NULLIF(SUM(Spend_EUR), 0) AS ROAS
    FROM df
    GROUP BY Platform, Campaign_Type
    ORDER BY Platform, Campaign_Type
""").df()

display(
    funnel_analysis.style.format({
        "Campaign_Count": "{:,.0f}",
        "Total_Spend_EUR": "€{:,.2f}",
        "Total_Platform_Conversions": "{:,.0f}",
        "Total_Platform_Revenue_EUR": "€{:,.2f}",
        "CPA": "€{:.2f}",
        "ROAS": "{:.2f}x"
    })
)

,Platform,Campaign_Type,Campaign_Count,Total_Spend_EUR,Total_Platform_Conversions,Total_Platform_Revenue_EUR,CPA,ROAS
0,DV360,Awareness,2,"€294,585.43","2,501","€229,849.33",€117.79,0.78x
1,DV360,Consideration,2,"€179,495.93","9,685","€1,044,251.04",€18.53,5.82x
2,DV360,Conversion,2,"€246,922.69","33,609","€5,517,380.37",€7.35,22.34x
3,Google Ads,Awareness,2,"€208,980.00","4,871","€511,586.19",€42.90,2.45x
4,Google Ads,Consideration,2,"€164,021.77","18,155","€2,090,224.39",€9.03,12.74x
5,Google Ads,Conversion,2,"€241,260.18","74,531","€10,710,485.44",€3.24,44.39x
6,Meta Ads,Awareness,2,"€186,293.68","9,640","€915,262.65",€19.33,4.91x
7,Meta Ads,Consideration,2,"€165,543.89","22,212","€2,487,063.19",€7.45,15.02x
8,Meta Ads,Conversion,2,"€269,375.46","76,525","€12,108,837.91",€3.52,44.95x
9,TikTok Ads,Awareness,2,"€157,145.33","7,377","€663,482.22",€21.30,4.22x


### Interpretation

Each platform has two campaigns per funnel stage.

Within Conversion campaigns, TikTok has the highest CPA (€8.24) and the lowest ROAS (15.13). DV360's Conversion CPA is €7.35, compared with €15.74 across all its campaign types.

The change in ranking demonstrates why funnel composition matters in platform comparisons.

Google Ads has the lowest Conversion CPA (€3.24), followed by Meta Ads (€3.52). This reverses their aggregate CPA ranking, where Meta Ads leads. Meta Ads retains the highest platform-reported Conversion ROAS at 44.95x, slightly above Google Ads at 44.39x.

## 6. Conversion campaign diagnostics

**Question:** Which metrics help explain differences in campaign CPA and ROAS?

I compare CPC, CVR and attributed revenue per conversion alongside CPA and ROAS.

For the same records and measurement source:

- CPA = CPC / CVR.
- ROAS = revenue per conversion / CPA.

These relationships describe the metric differences but do not establish their underlying causes.

In [26]:
conversion_analysis = duckdb.sql("""
    SELECT
        Platform,
        Campaign_ID,
        Campaign_Name,
        SUM(Spend_EUR) AS Total_Spend_EUR,
        SUM(Clicks) AS Total_Clicks,
        SUM(Platform_Conversions) AS Total_Platform_Conversions,
        SUM(Platform_Revenue_EUR) AS Total_Platform_Revenue_EUR,
        -- CPA = total spend ÷ total conversions
        SUM(Spend_EUR) / NULLIF(SUM(Platform_Conversions), 0) AS CPA,
        -- ROAS = total revenue ÷ total spend
        SUM(Platform_Revenue_EUR) / NULLIF(SUM(Spend_EUR), 0) AS ROAS,
        -- CVR = platform conversions ÷ clicks
        CAST(SUM(Platform_Conversions) AS DOUBLE) / NULLIF(SUM(Clicks), 0) AS CVR,
        -- CPC = total spend ÷ Clicks
        SUM(Spend_EUR) / NULLIF(SUM(Clicks), 0) AS CPC,
        -- Revenue per conversion = Platform_Revenue_EUR ÷ Platform_Conversions
        SUM(Platform_Revenue_EUR) / NULLIF(SUM(Platform_Conversions), 0) AS Revenue_per_conversion
    FROM df
    WHERE Campaign_Type = 'Conversion'
    GROUP BY Platform, Campaign_ID, Campaign_Name
    ORDER BY Platform, Total_Spend_EUR DESC
""").df()

display(
    conversion_analysis[
        [
            "Platform",
            "Campaign_ID",
            "Campaign_Name",
            "Total_Spend_EUR",
            "Total_Platform_Conversions",
            "CPC",
            "CPA",
            "CVR",
            "Revenue_per_conversion",
            "ROAS"
        ]
    ].style.format({
        "Total_Spend_EUR": "€{:,.2f}",
        "Total_Platform_Conversions": "{:,.0f}",
        "CPC": "€{:.2f}",
        "CPA": "€{:.2f}",
        "CVR": "{:.2%}",
        "Revenue_per_conversion": "€{:,.2f}",
        "ROAS": "{:.2f}x"
    })
)

,Platform,Campaign_ID,Campaign_Name,Total_Spend_EUR,Total_Platform_Conversions,CPC,CPA,CVR,Revenue_per_conversion,ROAS
0,DV360,DV_CV01,DV360 | Conversion | High Intent Retargeting,"€135,046.61","18,078",€0.33,€7.47,4.48%,€167.34,22.40x
1,DV360,DV_CV02,DV360 | Conversion | New Customer,"€111,876.08","15,531",€0.32,€7.20,4.39%,€160.47,22.28x
2,Google Ads,GGL_CV02,GGL | Conversion | Remarketing,"€131,483.30","40,059",€0.16,€3.28,4.96%,€141.98,43.26x
3,Google Ads,GGL_CV01,GGL | Conversion | High Intent,"€109,776.88","34,472",€0.16,€3.18,5.18%,€145.71,45.75x
4,Meta Ads,META_CV01,META | Conversion | Prospecting Purchases,"€142,753.23","38,969",€0.21,€3.66,5.62%,€156.79,42.80x
5,Meta Ads,META_CV02,META | Conversion | Retargeting,"€126,622.23","37,556",€0.20,€3.37,5.97%,€159.74,47.38x
6,TikTok Ads,TT_CV01,TT | Conversion | Prospecting,"€122,433.49","12,885",€0.20,€9.50,2.09%,€124.54,13.11x
7,TikTok Ads,TT_CV02,TT | Conversion | Retargeting,"€98,929.76","13,975",€0.21,€7.08,2.99%,€124.84,17.64x


### Interpretation

The two TikTok Conversion campaigns have CVRs of 2.09% and 2.99%, below the other Conversion campaigns in this dataset. Their attributed revenue per conversion is approximately €125, also lower than the other campaigns.

These factors help explain their lower ROAS. Lower revenue per conversion affects ROAS but does not directly affect CPA.

Within TikTok, TT_CV02 has a CPA of €7.08, slightly below both DV360 Conversion campaigns, while TT_CV01 has a CPA of €9.50.

## 7. Performance using the analytics source

**Question:** Do the campaign conclusions hold when conversions and revenue come from the separate analytics source?

I recalculate performance metrics using analytics-source conversions and revenue. Spend and clicks remain the same, so CPC is unchanged. Neither measurement source is assumed to be ground truth.

In [27]:
# SQL query to analyze conversion campaigns using Analytics-source metrics
analytics_conversion_analysis = duckdb.sql("""
    SELECT
        Platform,
        Campaign_ID,
        Campaign_Name,
        SUM(Spend_EUR) AS Total_Spend_EUR,
        SUM(Clicks) AS Total_Clicks,
        SUM(Analytics_Conversions) AS Total_Analytics_Conversions,
        SUM(Analytics_Revenue_EUR) AS Total_Analytics_Revenue_EUR,
        -- CPA = total spend ÷ total conversions (using Analytics data)
        SUM(Spend_EUR) / NULLIF(SUM(Analytics_Conversions), 0) AS Analytics_CPA,
        -- ROAS = total revenue ÷ total spend (using Analytics data)
        SUM(Analytics_Revenue_EUR) / NULLIF(SUM(Spend_EUR), 0) AS Analytics_ROAS,
        -- CVR = conversions ÷ clicks (using Analytics data)
        CAST(SUM(Analytics_Conversions) AS DOUBLE) / NULLIF(SUM(Clicks), 0) AS Analytics_CVR,
        -- CPC = total spend ÷ Clicks
        SUM(Spend_EUR) / NULLIF(SUM(Clicks), 0) AS CPC,
        -- Revenue per conversion = Analytics_Revenue_EUR ÷ Analytics_Conversions
        SUM(Analytics_Revenue_EUR) / NULLIF(SUM(Analytics_Conversions), 0) AS Analytics_Revenue_per_conversion
    FROM df
    WHERE Campaign_Type = 'Conversion'
    GROUP BY Platform, Campaign_ID, Campaign_Name
    ORDER BY Platform, Total_Spend_EUR DESC
""").df()

display(
    analytics_conversion_analysis.style.format({
        "Total_Spend_EUR": "€{:,.2f}",
        "Total_Clicks": "{:,.0f}",
        "Total_Analytics_Conversions": "{:,.0f}",
        "Total_Analytics_Revenue_EUR": "€{:,.2f}",
        "Analytics_CPA": "€{:.2f}",
        "Analytics_ROAS": "{:.2f}x",
        "Analytics_CVR": "{:.2%}",
        "CPC": "€{:.2f}",
        "Analytics_Revenue_per_conversion": "€{:,.2f}"
    })
)

,Platform,Campaign_ID,Campaign_Name,Total_Spend_EUR,Total_Clicks,Total_Analytics_Conversions,Total_Analytics_Revenue_EUR,Analytics_CPA,Analytics_ROAS,Analytics_CVR,CPC,Analytics_Revenue_per_conversion
0,DV360,DV_CV01,DV360 | Conversion | High Intent Retargeting,"€135,046.61","403,190","16,884","€2,826,581.36",€8.00,20.93x,4.19%,€0.33,€167.41
1,DV360,DV_CV02,DV360 | Conversion | New Customer,"€111,876.08","353,798","14,548","€2,332,709.72",€7.69,20.85x,4.11%,€0.32,€160.35
2,Google Ads,GGL_CV02,GGL | Conversion | Remarketing,"€131,483.30","808,179","39,923","€5,669,785.45",€3.29,43.12x,4.94%,€0.16,€142.02
3,Google Ads,GGL_CV01,GGL | Conversion | High Intent,"€109,776.88","665,842","34,385","€5,007,481.30",€3.19,45.62x,5.16%,€0.16,€145.63
4,Meta Ads,META_CV01,META | Conversion | Prospecting Purchases,"€142,753.23","693,181","36,570","€5,729,017.03",€3.90,40.13x,5.28%,€0.21,€156.66
5,Meta Ads,META_CV02,META | Conversion | Retargeting,"€126,622.23","629,362","35,290","€5,642,498.63",€3.59,44.56x,5.61%,€0.20,€159.89
6,TikTok Ads,TT_CV01,TT | Conversion | Prospecting,"€122,433.49","617,495","12,885","€1,604,800.30",€9.50,13.11x,2.09%,€0.20,€124.55
7,TikTok Ads,TT_CV02,TT | Conversion | Retargeting,"€98,929.76","468,116","13,975","€1,743,582.64",€7.08,17.62x,2.99%,€0.21,€124.76


### Interpretation

TikTok retains the lowest Conversion campaign ROAS and CVR in this comparison.

Measurement source selection changes other reported results. For example, META_CV01 has a platform-reported ROAS of 42.80 versus 40.13 using the analytics source across the full analysis period.

The following analysis examines when these differences emerge.

## 8. Measurement reconciliation over time

### 8.1 Weekly comparison across platforms

**Question:** Are conversion discrepancies stable or concentrated in particular weeks?

The percentage gap is calculated as `(platform conversions − analytics conversions) / analytics conversions × 100`. Positive values mean the platform reports more conversions.

The week beginning 31 August contains only one day of this dataset and is treated as a partial week.

In [28]:
# SQL query to analyze the weekly conversion gap between Platform and Analytics
weekly_conversion_gap = duckdb.sql("""
    SELECT
        DATE_TRUNC('week', Date) AS Week,
        Platform,
        SUM(Platform_Conversions) AS Total_Platform_Conversions,
        SUM(Analytics_Conversions) AS Total_Analytics_Conversions,
        -- Conversion gap % = (Platform conversions - Analytics) / Analytics conversions * 100
        (SUM(Platform_Conversions) - SUM(Analytics_Conversions)) * 100.0 / NULLIF(SUM(Analytics_Conversions), 0) AS Conversion_Gap_Percentage
    FROM df
    WHERE Campaign_Type = 'Conversion'
    GROUP BY Week, Platform
    ORDER BY Week, Platform
""").df()

display(
    weekly_conversion_gap.style.format({
        "Week": "{:%Y-%m-%d}",
        "Total_Platform_Conversions": "{:,.0f}",
        "Total_Analytics_Conversions": "{:,.0f}",
        "Conversion_Gap_Percentage": "{:.2f}%"
    })
)

,Week,Platform,Total_Platform_Conversions,Total_Analytics_Conversions,Conversion_Gap_Percentage
0,2026-06-01,DV360,"2,643","2,476",6.74%
1,2026-06-01,Google Ads,"5,760","5,760",0.00%
2,2026-06-01,Meta Ads,"5,653","5,653",0.00%
3,2026-06-01,TikTok Ads,"2,095","2,095",0.00%
4,2026-06-08,DV360,"2,658","2,493",6.62%
5,2026-06-08,Google Ads,"5,700","5,700",0.00%
6,2026-06-08,Meta Ads,"5,672","5,672",0.00%
7,2026-06-08,TikTok Ads,"2,081","2,081",0.00%
8,2026-06-15,DV360,"2,605","2,434",7.03%
9,2026-06-15,Google Ads,"5,731","5,731",0.00%


DV360 shows a persistent conversion gap of approximately 6–7%, while Google Ads shows a temporary gap that reaches 3.96% in the week beginning 10 August. TikTok reports identical conversion counts across sources throughout the period.

Meta Ads is selected for deeper investigation because its gap changes abruptly from zero to a sustained level of approximately 20–22% during the complete weeks of August.

### 8.2 Weekly comparison within Meta Ads

The platform-level analysis identifies an increase in the Meta conversion gap in the week beginning 27 July.

**Question:** Does the change affect both Meta Conversion campaigns during the same week?

I separate META_CV01 and META_CV02 to determine whether the pattern is shared or campaign-specific.

In [29]:
# SQL query to analyze the weekly conversion gap between Platform and Analytics specifically for Meta Ads conversion campaigns
weekly_conversion_gap_meta = duckdb.sql("""
    SELECT
        DATE_TRUNC('week', Date) AS Week,
        Platform,
        Campaign_ID,
        SUM(Platform_Conversions) AS Total_Platform_Conversions,
        SUM(Analytics_Conversions) AS Total_Analytics_Conversions,
        -- Conversion gap % = (Platform conversions - Analytics) / Analytics conversions * 100
        (SUM(Platform_Conversions) - SUM(Analytics_Conversions)) * 100.0 / NULLIF(SUM(Analytics_Conversions), 0) AS Conversion_Gap_Percentage
    FROM df
    WHERE Campaign_Type = 'Conversion'
      AND Platform = 'Meta Ads'
    GROUP BY Week, Platform, Campaign_ID
    ORDER BY Week, Platform, Campaign_ID
""").df()

display(
    weekly_conversion_gap_meta.style.format({
        "Week": "{:%Y-%m-%d}",
        "Total_Platform_Conversions": "{:,.0f}",
        "Total_Analytics_Conversions": "{:,.0f}",
        "Conversion_Gap_Percentage": "{:.2f}%"
    })
)

,Week,Platform,Campaign_ID,Total_Platform_Conversions,Total_Analytics_Conversions,Conversion_Gap_Percentage
0,2026-06-01,Meta Ads,META_CV01,"2,900","2,900",0.00%
1,2026-06-01,Meta Ads,META_CV02,"2,753","2,753",0.00%
2,2026-06-08,Meta Ads,META_CV01,"2,875","2,875",0.00%
3,2026-06-08,Meta Ads,META_CV02,"2,797","2,797",0.00%
4,2026-06-15,Meta Ads,META_CV01,"2,952","2,952",0.00%
5,2026-06-15,Meta Ads,META_CV02,"2,887","2,887",0.00%
6,2026-06-22,Meta Ads,META_CV01,"3,071","3,071",0.00%
7,2026-06-22,Meta Ads,META_CV02,"2,843","2,843",0.00%
8,2026-06-29,Meta Ads,META_CV01,"2,958","2,958",0.00%
9,2026-06-29,Meta Ads,META_CV02,"2,876","2,876",0.00%


### 8.3 Daily investigation

Both Meta Conversion campaigns show a gap in the week beginning 27 July, followed by gaps of approximately 20–22% during the complete weeks of August.

**Question:** On which day does the conversion discrepancy begin, and does attributed revenue diverge at the same time?

I examine daily conversion and revenue gaps from 20 July to 10 August.

In [30]:
# SQL query to generate a daily report for specific Meta Ads campaigns, including conversion and revenue gaps
daily_gap_report = duckdb.sql("""
    SELECT
        Date,
        Campaign_ID,
        SUM(Platform_Conversions) AS Total_Platform_Conversions,
        SUM(Analytics_Conversions) AS Total_Analytics_Conversions,
        -- Conversion gap %
        (SUM(Platform_Conversions) - SUM(Analytics_Conversions)) * 100.0 / NULLIF(SUM(Analytics_Conversions), 0) AS Conversion_Gap_Percentage,
        SUM(Platform_Revenue_EUR) AS Total_Platform_Revenue,
        SUM(Analytics_Revenue_EUR) AS Total_Analytics_Revenue,
        -- Revenue gap %
        (SUM(Platform_Revenue_EUR) - SUM(Analytics_Revenue_EUR)) * 100.0 / NULLIF(SUM(Analytics_Revenue_EUR), 0) AS Revenue_Gap_Percentage
    FROM df
    WHERE Campaign_ID IN ('META_CV01', 'META_CV02')
      AND Date BETWEEN '2026-07-20' AND '2026-08-10'
    GROUP BY Date, Campaign_ID
    ORDER BY Date, Campaign_ID
""").df()

display(
    daily_gap_report.style.format({
        "Date": "{:%Y-%m-%d}",
        "Total_Platform_Conversions": "{:,.0f}",
        "Total_Analytics_Conversions": "{:,.0f}",
        "Conversion_Gap_Percentage": "{:.2f}%",
        "Total_Platform_Revenue": "€{:,.2f}",
        "Total_Analytics_Revenue": "€{:,.2f}",
        "Revenue_Gap_Percentage": "{:.2f}%"
    })
)

,Date,Campaign_ID,Total_Platform_Conversions,Total_Analytics_Conversions,Conversion_Gap_Percentage,Total_Platform_Revenue,Total_Analytics_Revenue,Revenue_Gap_Percentage
0,2026-07-20,META_CV01,385,385,0.00%,"€60,576.20","€60,942.28",-0.60%
1,2026-07-20,META_CV02,483,483,0.00%,"€79,740.36","€78,843.75",1.14%
2,2026-07-21,META_CV01,471,471,0.00%,"€75,611.00","€75,654.43",-0.06%
3,2026-07-21,META_CV02,410,410,0.00%,"€65,684.31","€65,529.34",0.24%
4,2026-07-22,META_CV01,484,484,0.00%,"€77,278.74","€77,115.87",0.21%
5,2026-07-22,META_CV02,420,420,0.00%,"€65,231.15","€65,272.78",-0.06%
6,2026-07-23,META_CV01,474,474,0.00%,"€74,045.99","€74,373.58",-0.44%
7,2026-07-23,META_CV02,500,500,0.00%,"€81,379.76","€80,026.91",1.69%
8,2026-07-24,META_CV01,431,431,0.00%,"€66,224.03","€65,965.27",0.39%
9,2026-07-24,META_CV02,417,417,0.00%,"€66,009.86","€66,542.58",-0.80%


### Interpretation

The conversion discrepancy begins on 31 July in both Meta campaigns, alongside a substantial increase in the revenue gap.

The simultaneous change supports investigating a shared measurement or attribution factor. The dataset does not establish which source is correct or identify the cause.

## 9. Impact on reported performance

**Question:** How much does measurement source selection change reported results after the divergence begins?

For 31 July–31 August, I compare conversion counts, attributed revenue and ROAS for both Meta campaigns. Gaps retain their sign and are defined as platform totals minus analytics totals.

In [31]:
# Compare signed measurement gaps and ROAS from 31 July to 31 August.
meta_measurement_impact = duckdb.sql("""
    SELECT
        Campaign_ID,
        SUM(Platform_Conversions) AS Platform_Conversions,
        SUM(Analytics_Conversions) AS Analytics_Conversions,
        -- Gap = platform total minus analytics total; positive means platform reports more
        SUM(Platform_Conversions) - SUM(Analytics_Conversions) AS Conversion_Gap,
        SUM(Platform_Revenue_EUR) AS Platform_Revenue_EUR,
        SUM(Analytics_Revenue_EUR) AS Analytics_Revenue_EUR,
        -- Gap = platform total minus analytics total; positive means platform reports more
        SUM(Platform_Revenue_EUR) - SUM(Analytics_Revenue_EUR) AS Revenue_Gap_EUR,
        -- Side-by-side ROAS
        SUM(Platform_Revenue_EUR) / NULLIF(SUM(Spend_EUR), 0) AS Platform_ROAS,
        SUM(Analytics_Revenue_EUR) / NULLIF(SUM(Spend_EUR), 0) AS Analytics_ROAS
    FROM df
    WHERE Date BETWEEN '2026-07-31' AND '2026-08-31'
    AND Campaign_ID IN ('META_CV01', 'META_CV02')
    GROUP BY Campaign_ID
    ORDER BY Campaign_ID
""").df()

display(
    meta_measurement_impact.style.format({
        "Platform_Conversions": "{:,.0f}",
        "Analytics_Conversions": "{:,.0f}",
        "Conversion_Gap": "{:,.0f}",
        "Platform_Revenue_EUR": "€{:,.2f}",
        "Analytics_Revenue_EUR": "€{:,.2f}",
        "Revenue_Gap_EUR": "€{:,.2f}",
        "Platform_ROAS": "{:.2f}x",
        "Analytics_ROAS": "{:.2f}x"
    })
)

,Campaign_ID,Platform_Conversions,Analytics_Conversions,Conversion_Gap,Platform_Revenue_EUR,Analytics_Revenue_EUR,Revenue_Gap_EUR,Platform_ROAS,Analytics_ROAS
0,META_CV01,"13,658","11,259","2,399","€2,138,952.32","€1,758,727.53","€380,224.79",43.91x,36.10x
1,META_CV02,"12,973","10,707","2,266","€2,059,234.91","€1,701,557.58","€357,677.33",47.18x,38.99x


### Interpretation

META_CV01 reports 2,399 more conversions and €380,224.79 more attributed revenue in the platform source. Its ROAS is 43.91 using platform data versus 36.10 using analytics data.

META_CV02 reports 2,266 more conversions and €357,677.33 more attributed revenue in the platform source. Its ROAS is 47.18 versus 38.99.

These differences quantify how measurement source selection affects reported performance. They do not represent proven revenue loss or incremental sales.

## 10. Recommendations and limitations

### Recommended next actions

- Review measurement changes affecting the Meta campaigns around 31 July before using platform-reported performance to justify additional investment.
- Reconcile event definitions, attribution windows and deduplication rules across sources.
- Investigate TikTok conversion efficiency by campaign, device and landing-page performance before recommending budget changes.
- Present platform and analytics-source results with explicit source labels in the dashboard.

### Limitations

All data is synthetic. Spend was revised during dataset preparation, which affects cost-based metrics and ROAS.

The analysis is descriptive. It does not establish causality, incremental revenue, profitability or the return from increasing spend. Event-level records and configuration history would be needed to investigate the cause of the measurement discrepancy.

Conversions and attributed revenue may overlap across advertising platforms. Without cross-platform deduplication, their combined totals should not be interpreted as unique business conversions or revenue.

### Interactive dashboard

[View the Looker Studio dashboard](https://datastudio.google.com/s/nxXR821mqs4)

The dashboard covers performance overview, conversion diagnostics
and measurement reconciliation. Its metrics use aggregated totals
and respond to the selected filters.

## 11. Dashboard data export

Export the validated data at date–campaign–device level for Looker Studio.

Spend, traffic metrics and source-specific conversions and revenue remain numeric.
Performance ratios will be calculated from aggregated totals in the dashboard
so they respond correctly to filters.

The repeated campaign-day budget is excluded from this export to prevent
double counting across devices.

In [32]:
# Prepare the dashboard dataset at date-campaign-device level
dashboard_data = duckdb.sql("""
    SELECT
        Date,
        Platform,
        Campaign_ID,
        Campaign_Name,
        Campaign_Type,
        Device,
        Spend_EUR,
        Impressions,
        Clicks,
        Landing_Page_Views,
        Platform_Conversions,
        Analytics_Conversions,
        Platform_Revenue_EUR,
        Analytics_Revenue_EUR
    FROM df
    ORDER BY Date, Platform, Campaign_ID, Device
""").df()

# Save the dashboard export in the temporary Colab session
export_path = Path("/content/marketing_dashboard_jun_aug_2026.csv")

dashboard_data.to_csv(
    export_path,
    index=False,
    date_format="%Y-%m-%d"
)

print("Exported rows:", len(dashboard_data))
print("Exported columns:", dashboard_data.shape[1])
print("File saved:", export_path)

Exported rows: 6624
Exported columns: 14
File saved: /content/marketing_dashboard_jun_aug_2026.csv
